In [ ]:
import jax
import jax.numpy as jnp
import sys
# sys.path.append('/Users/julien.tang/Desktop/code_development/atmosphere/atmo3/atmo3')
from atmo3 import grid_utils

## Define the grid parameters and the layers

In [2]:
site_altitude = 5000.0
N = 300
Lbox = 20000.0

Initialize the grid

In [3]:
grid_space = grid_utils.GridWorkspace(
    N=N, Lbox=Lbox, site_altitude=site_altitude
)

Define insrument static position and pointing

In [4]:
theta = 60.0  # degrees
phi = 45.0    # degrees
gamma_tol = 5.0  # degrees
tolerance = jnp.cos(jnp.deg2rad(gamma_tol))
detector_position = jnp.array([4/7*grid_space.Lbox, 2/7*grid_space.Lbox, site_altitude])
unit_pointing_vec_reference = grid_space.lonlat_to_unitvec(theta, phi, lonlat=True)

## Plotting utils

In [5]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [6]:
def add_axis(fig, start, direction, length, color='black', width=8, arrow_frac=.1):
    """
    Add a 3D axis line to a Plotly figure.
    
    Parameters
    ----------
    fig : go.Figure
        The Plotly 3D figure.
    start : tuple of 3 floats
        (x, y, z) starting point of the axis.
    direction : str
        Axis direction: 'x', 'y', or 'z'.
    length : float
        Length of the axis line.
    color : str, optional
        Line color. Default is 'black'.
    width : float, optional
        Line width. Default is 8.
    arrow_frac : float, optional
        Fraction of the line to allocate for arrowhead.
    """
    x_det, y_det, z_det = start

    

    if direction == 'x':
        x = [x_det, x_det + length]
        y = [y_det, y_det]
        z = [z_det, z_det]
        unit_vec = jnp.array([1,0,0])
    elif direction == 'y':
        x = [x_det, x_det]
        y = [y_det, y_det + length]
        z = [z_det, z_det]
        unit_vec = jnp.array([0,1,0])
    elif direction == 'z':
        x = [x_det, x_det]
        y = [y_det, y_det]
        z = [z_det, z_det + length]
        unit_vec = jnp.array([0,0,1])
    else:
        raise ValueError(f"Unknown direction: {direction}")
    
    norm_vec = unit_vec/jnp.linalg.norm(unit_vec)*length

    tip = jnp.array([x_det, y_det, z_det]) + norm_vec
    
    fig.add_trace(go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode='lines',
        line=dict(color=color, width=width),
        showlegend=False
    ))

    # Arrowhead: small lines forming a "V"
    arrow_len = length * arrow_frac
    # Pick two perpendicular vectors to direction
    if jnp.allclose(norm_vec[:2], 0):
        perp1 = jnp.array([1,0,0])
    else:
        perp1 = jnp.array([-norm_vec[1], norm_vec[0], 0])
    perp1 = perp1 / jnp.linalg.norm(perp1) * arrow_len

    perp2 = jnp.cross(norm_vec, perp1)
    perp2 = perp2 / jnp.linalg.norm(perp2) * arrow_len

    for vec in [perp1, -perp1, perp2, -perp2]:
        fig.add_trace(go.Scatter3d(
            x=[tip[0], tip[0]-vec[0]],
            y=[tip[1], tip[1]-vec[1]],
            z=[tip[2], tip[2]-vec[2]],
            mode='lines',
            line=dict(color=color, width=width),
            showlegend=False
        ))




In [7]:
def add_cone_mesh(fig, det_pos, unit_pointing_vec_reference, gamma_tol,
                  cone_length=500.0, n_circle=16, n_length=10,
                  color='blue', opacity=0.12, name='Cone'):
    """
    Add a triangular mesh representing a cone (field-of-view) to a Plotly Figure.

    Parameters
    ----------
    fig : plotly.graph_objects.Figure
        Figure to draw the mesh into (modified in-place).
    det_pos : array-like (3,)
        Detector position in the same coordinate system as the cone (i,j,k or meters).
    unit_pointing_vec_reference : array-like (3,)
        Unit vector for pointing direction.
    gamma_tol : float
        Cone half-angle in degrees.
    cone_length : float, optional
        Length of the cone (default: 500.0).
    n_circle : int, optional
        Number of points around each circular layer (default: 16).
    n_length : int, optional
        Number of layers along the cone length (default: 10).
    color : str, optional
        Mesh color.
    opacity : float, optional
        Mesh opacity.
    name : str, optional
        Legend name for the mesh.

    Returns
    -------
    verts : ndarray
        Array of vertices shape (n_circle*n_length, 3)
    faces : ndarray
        Array of triangle faces shape (n_triangles, 3)
    """
    det_pos = jnp.asarray(det_pos)
    u = jnp.asarray(unit_pointing_vec_reference)

    # choose a reference vector not parallel to u
    ref = jnp.array([0, 0, 1])
    if jnp.isclose(jnp.abs(jnp.dot(u, ref)), 1.0):
        ref = jnp.array([0, 1, 0])

    e1 = jnp.cross(u, ref)
    e1 = e1 / jnp.linalg.norm(e1)
    e2 = jnp.cross(u, e1)

    t_vals = jnp.linspace(0, cone_length, n_length)
    phi_vals = jnp.linspace(0, 2*jnp.pi, n_circle, endpoint=False)

    vertices = []
    for t in t_vals:
        R = t * jnp.tan(jnp.deg2rad(gamma_tol))
        for phi in phi_vals:
            point = det_pos + u * t + R * (jnp.cos(phi) * e1 + jnp.sin(phi) * e2)
            vertices.append(point)

    vertices = jnp.asarray(vertices)
    x, y, z = vertices.T

    # build faces connecting layer i to i+1
    faces = []
    for i in range(n_length - 1):
        for j in range(n_circle):
            j_next = (j + 1) % n_circle
            a = i * n_circle + j
            b = i * n_circle + j_next
            c = (i + 1) * n_circle + j
            d = (i + 1) * n_circle + j_next
            faces.append([a, b, c])
            faces.append([b, d, c])
    faces = jnp.asarray(faces)

    # Add mesh to figure
    fig.add_trace(go.Mesh3d(
        x=x, y=y, z=z,
        i=faces[:, 0].astype(int),
        j=faces[:, 1].astype(int),
        k=faces[:, 2].astype(int),
        color=color,
        opacity=opacity,
        name=name,
        hoverinfo='skip',
        showscale=False
    ))

    return vertices, faces


# Example usage (keeps existing variables from the notebook):
# verts, faces = add_cone_mesh(fig, (det_pos_i, det_pos_j, det_pos_k), unit_pointing_vec_reference, gamma_tol)
# fig.show()

In [9]:
cone_length = 1.3*grid_space.N
n_circle = 16      # number of points around circle
n_length = 10      # number of layers along the cone

# Build local orthonormal frame for cone base
ref = jnp.array([0, 0, 1])

## Compute Line of sight voxels

In [8]:
detector_position_ijk = detector_position / grid_space.grid_spacing
det_pos_i, det_pos_j, det_pos_k = detector_position_ijk

# Tada!

In [10]:
voxels_indices_slices = grid_space.zslice_to_voxels(
    unit_pointing_vec_reference=unit_pointing_vec_reference,
    detector_position=detector_position,
    tolerance=tolerance
)

In [14]:
# Create interactive 3D plot with Plotly
fig = go.Figure()

# Step 5: flatten and keep only non-zero centers
# A voxel is non-zero if at least one coordinate > 0
nonzero_mask = jnp.any(voxels_indices_slices > 0, axis=0)
# Extract the selected voxel centers
selected_voxels = voxels_indices_slices[:,nonzero_mask]  # shape (num_selected, 3)

fig.add_trace(go.Scatter3d(
    x=selected_voxels[0], y=selected_voxels[1], z=selected_voxels[2],
    mode='markers',
    marker=dict(size=.3, color='black', opacity=0.5),
    name='Center of voxels within separation angle',
    showlegend=True
))

# Add detector position
fig.add_trace(go.Scatter3d(
    x= [det_pos_i], y= [det_pos_j], z= [det_pos_k],
    mode='markers',
    marker=dict(size=6, color='red', symbol='diamond'),
    name='Detector',
    showlegend=True
))

# Add detector coordinate axes
axis_len = N//10.


add_axis(fig, (det_pos_i, det_pos_j, det_pos_k), 'x', axis_len, color='black')
add_axis(fig, (det_pos_i, det_pos_j, det_pos_k), 'y', axis_len, color='black')
add_axis(fig, (det_pos_i, det_pos_j, det_pos_k), 'z', axis_len, color='black')

add_cone_mesh(
    fig,
    det_pos=(det_pos_i, det_pos_j, det_pos_k),
    unit_pointing_vec_reference=unit_pointing_vec_reference,
    gamma_tol=gamma_tol,
    cone_length=cone_length,
    n_circle=n_circle,
    n_length=n_length,
    color='royalblue',
    opacity=0.12,
    name=r'Cone of separation angle $\gamma = $' + str(gamma_tol) + '°'
)

# Update layout for better visualization
fig.update_layout(
    scene=dict(
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        xaxis=dict(range=[0, grid_space.N]),
        yaxis=dict(range=[0, grid_space.N]),
        zaxis=dict(range=[0, grid_space.N]),
        aspectmode='cube'
    ),
    title='Interactive 3D Visualization: Detector + Voxel Grid + Line-of-Sight Points in a given layer',
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

# Show the plot
fig.show()